In [3]:
import numpy as np
import scipy.stats as stats
import random as rng

In [22]:
def attack(k=16, p=5000, S_start= 200_000_000, S_end=900_000_000, mu1 = 6000, mu2 = 6000, stdev = np.sqrt(1000), n = 1_000_000, num_trials = 2000, seed = None):
    rng = np.random.default_rng(seed)
    U = rng.exponential(scale=mu1, size=k)                                                            # edit the k value.                    # public set, fixed for this run - If you want the same U for every
    V = rng.normal(mu2, stdev, size=k)
    eve_list_mu1 = []
    eve_list_mu2 = []
    for _ in range(num_trials):

        bob_pick = U if rng.random() < 0.5 else V
        bob_sum = rng.choice(bob_pick, size = p).sum() #bob flips a coin to determine his selection and then oversamples from parent set U or V.
        S = rng.uniform(S_start, S_end)
        bob_tx = bob_sum + S                                  # Bob -> Alice (S masks bob_sum)


        alice_sum_U = rng.exponential(scale=mu1, size=n-p).sum() + bob_tx
        alice_sum_V = rng.normal(mu2, stdev, size=n-p).sum() + bob_tx #note that Alice creates both sets

        N_1 = alice_sum_U/n - mu1 #Alice calculates what to send back to Bob
        N_2 = alice_sum_V/n - mu2 #Alice ---> Bob

        eve_val = bob_tx/n

        eve_result_1, eve_result_2 = abs(N_1 - eve_val), abs(N_2 - eve_val)

        mu1_approx, mu2_approx = eve_result_1 * (n/p), eve_result_2 * (n/p)
        
        eve_list_mu1.append(mu1_approx)
        eve_list_mu2.append(mu2_approx)

    return {
            "Mu1 approximations": eve_list_mu1[:20],
            "mean of mu1 approx": np.mean(eve_list_mu1),
            "Mu2 approximations": eve_list_mu2[:20],
            "mean of mu2 approx": np.mean(eve_list_mu2),
            "mu1": mu1,
            "mu2": mu2,
            "U": U,
            "V": V
        }

    

In [25]:
attack(stdev=np.sqrt(1_000_000), num_trials=1000)

{'Mu1 approximations': [np.float64(5947.892589715957),
  np.float64(6113.341334421921),
  np.float64(5904.375948618417),
  np.float64(7020.649856509033),
  np.float64(7039.310313069996),
  np.float64(5843.824303870292),
  np.float64(4941.496594701016),
  np.float64(7050.574677151701),
  np.float64(7462.376170828566),
  np.float64(5789.766799348467)],
 'mean of mu1 approx': np.float64(6032.7216011929495),
 'Mu2 approximations': [np.float64(6084.368012791674),
  np.float64(5739.819404815671),
  np.float64(5971.58492659064),
  np.float64(5990.9501529404115),
  np.float64(5936.322918985002),
  np.float64(5714.875843409004),
  np.float64(5937.138324050716),
  np.float64(6160.939619714838),
  np.float64(6370.434210835219),
  np.float64(5738.201746394657)],
 'mean of mu2 approx': np.float64(5996.958574390629),
 'mu1': 6000,
 'mu2': 6000,
 'U': array([ 1118.85777874,  4446.27821295,  1960.87617091,  3218.8126689 ,
        16753.63316615, 13164.08262188,  2498.9002082 ,  4066.07950682,
        